Encoder-only models like BERT tend to be great at extracting answers to factoid questions like “Who invented the Transformer architecture?” but fare poorly when given open-ended questions like “Why is the sky blue?” In these more challenging cases, encoder-decoder models like T5 and BART are typically used to synthesize the information in a way that’s quite similar to text summarization.

# This script will fine turn Bert in SQuAD dataset

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

# Preparing dataset

In [ ]:
from datasets import load_dataset
raw_datasets = load_dataset("squad")
raw_datasets

In [ ]:
raw_datasets["train"].filter(lambda x : len(x["answers"]["text"]) !=1)

In [ ]:
print(raw_datasets["validation"][0]["answers"])

# Precessing dataset

In [ ]:
from transformers import AutoTokenizer
model_checkpoint = "bert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

overflow_to_sample_mapping have function: a mapping from each chunk → its original data sample

In [ ]:
# test

question = raw_datasets["train"][2:6]["question"]
context =  raw_datasets["train"][2:6]["context"]
inputs = tokenizer(
    question,
    context,
    max_length=100,
    truncation="only_second",
    stride=50,
    return_overflowing_tokens=True,
    return_offsets_mapping=True,

)


# # for ids  in inputs["input_ids"]:
#   print(tokenizer.decode(ids))

# # inputs.keys()
# print(f"The 4 examples gave {len(inputs['input_ids'])} features.")
# print(f"Here is where each comes from: {inputs['overflow_to_sample_mapping']}.")

# ressult will is:
# The 4 examples gave 19 features.
# Here is where each comes from: [0, 0, 0, 0, 1, 1, 1, 1, 2, 2, 2, 2, 3, 3, 3, 3, 3, 3, 3].


# find start position and end position corresponding
answers = raw_datasets["train"][2:6]["answers"]
start_positions =[]
end_positions = []
for i, offset in enumerate(inputs["offset_mapping"]):
  sample_idx = inputs["overflow_to_sample_mapping"][i]
  answer = answers[sample_idx]
  start_char = answer["answer_start"][0]
  end_char = answer["answer_start"][0] + len(answer["text"][0])
  sequence_ids = inputs.sequence_ids(i)

  # find start and  end of context
  idx = 0
  while sequence_ids[idx] != 1:
    idx+=1
  context_start = idx
  while sequence_ids[idx] ==1:
    idx+=1
  context_end = idx-1
  if offset[context_start][0] > start_char or offset[context_end][1] < end_char:
        start_positions.append(0)
        end_positions.append(0)
  else:
      # Otherwise it's the start and end token positions
      idx = context_start
      while idx <= context_end and offset[idx][0] <= start_char:
          idx += 1
      start_positions.append(idx - 1)

      idx = context_end
      while idx >= context_start and offset[idx][1] >= end_char:
          idx -= 1
      end_positions.append(idx + 1)

# start_positions, end_positions

# ([83, 51, 19, 0, 0, 64, 27, 0, 34, 0, 0, 0, 67, 34, 0, 0, 0, 0, 0],
#  [85, 53, 21, 0, 0, 70, 33, 0, 40, 0, 0, 0, 68, 35, 0, 0, 0, 0, 0])


# check
idx = 4
sample_idx = inputs["overflow_to_sample_mapping"][idx]
answer = answers[sample_idx]["text"][0]


start = start_positions[idx]
end = end_positions[idx]

label_answers = tokenizer.decode(inputs["input_ids"][idx][start:end+1])
# print(f"answers: {answer}, labels: {label_answers}")

decoded_example = tokenizer.decode(inputs["input_ids"][idx])
print(f"Theoretical answer: {answer}, decoded example: {decoded_example}")

In [ ]:
max_length = 384
stride = 128
def preprocess_training_example(examples):
  inputs = tokenizer(
      examples["question"],
      examples["context"],
      max_length = max_length,
      truncation = True,
      stride = stride,
      return_overflowing_tokens = True,
      return_offsets_mapping = True,
      padding = "max_length"
  )

  offset_mapping = inputs.pop("offset_mapping")
  sample_map = inputs.pop("overflow_to_sample_mapping")
  answers = examples["answers"]
  start_positions = []
  end_positions = []


  for i, offset in enumerate(offset_mapping):
    sample_idx = sample_map[i]
    answer = answers[sample_idx]
    start_char = answer["answer_start"][0]
    end_char = answer["answer_start"][0] + len(answer["text"][0])
    sequence_idx = inputs.sequence_ids(i)
    idx = 0
    while sequence_ids[idx] != 1:
      idx += 1
    context_start = idx
    while sequence_ids[idx] == 1:
      idx += 1
    context_end = idx - 1

    # If the answer is not fully inside the context, label is (0, 0)
    if offset[context_start][0] > start_char or offset[context_end][1] < end_char:
        start_positions.append(0)
        end_positions.append(0)
    else:
        # Otherwise it's the start and end token positions
        idx = context_start
        while idx <= context_end and offset[idx][0] <= start_char:
            idx += 1
        start_positions.append(idx - 1)

        idx = context_end
        while idx >= context_start and offset[idx][1] >= end_char:
            idx -= 1
        end_positions.append(idx + 1)

  inputs["start_positions"] = start_positions
  inputs["end_positions"] = end_positions
  return inputs


train_dataset = raw_datasets["train"].map(
    preprocess_training_example, batched = True, remove_columns = raw_datasets["train"].column_names
)
len(raw_datasets["train"]), len(train_dataset)

# Preprocessing val dataset

comppute validation loss won't help us understand how good the model is

In [ ]:
n_best = 20
max_answer_length = 30
predicted_answers = []
def preprocess_vali_examples(examples):
    questions = [q.strip() for q in examples["question"]]
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=max_length,
        truncation="only_second",
        stride=stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_map = inputs.pop("overflow_to_sample_mapping")
    example_ids = []

    for i in range(len(inputs["input_ids"])):
        sample_idx = sample_map[i]
        example_ids.append(examples["id"][sample_idx])

        sequence_ids = inputs.sequence_ids(i) # list token to know where is the part token from
        # offset will have contain of question and context
        offset = inputs["offset_mapping"][i]

        inputs["offset_mapping"][i] = [
            # 0: belong to quetions, 1: belong to context, None special token([CLS], [SEP], padding)
            o if sequence_ids[k] == 1 else None for k, o in enumerate(offset)
        ]

    inputs["example_id"] = example_ids
    return inputs
validation_dataset = raw_datasets["validation"].map(
    preprocess_vali_examples,
    batched=True,
    remove_columns=raw_datasets["validation"].column_names,
)
len(raw_datasets["validation"]), len(validation_dataset)

# Fine-tuning the model with the Trainer API

post Processing

Action we will took:
<br>
marsked the start and end correspongind  to tokens outside of the context
<br>
using softmax to converted start and enc logits
<br>
attributed a score to each (start_token, end_token) pair by taking the product of the corresponding two probabilities.
<br>
maximum score that yielded a valid answer

!pip install evaluate

In [ ]:
!pip install evaluate

compute_metrics() function only receives a tuple eval_preds with logits and labels. Here we will need a bit more, as we have to look in the dataset of features for the offset and in the dataset of examples for the original contexts, so we won’t be able to use this function to get regular evaluation results during training. We will only use it at the end of training to check the results.

In [ ]:
from tqdm.auto import tqdm
import evaluate
import collections
import numpy as np
metric = evaluate.load("squad")

def compute_metrics(start_logits, end_logits, features, examples):
    example_to_features = collections.defaultdict(list)
    for idx, feature in enumerate(features):
        example_to_features[feature["example_id"]].append(idx)

    predicted_answers = []
    for example in tqdm(examples):
        example_id = example["id"]
        context = example["context"]
        answers = []

        # Loop through all features associated with that example
        for feature_index in example_to_features[example_id]:
            start_logit = start_logits[feature_index]
            end_logit = end_logits[feature_index]
            offsets = features[feature_index]["offset_mapping"]

            start_indexes = np.argsort(start_logit)[-1 : -n_best - 1 : -1].tolist()
            end_indexes = np.argsort(end_logit)[-1 : -n_best - 1 : -1].tolist()
            for start_index in start_indexes:
                for end_index in end_indexes:
                    # Skip answers that are not fully in the context
                    if offsets[start_index] is None or offsets[end_index] is None:
                        continue
                    # Skip answers with a length that is either < 0 or > max_answer_length
                    if (
                        end_index < start_index
                        or end_index - start_index + 1 > max_answer_length
                    ):
                        continue

                    answer = {
                        "text": context[offsets[start_index][0] : offsets[end_index][1]],
                        "logit_score": start_logit[start_index] + end_logit[end_index],
                    }
                    answers.append(answer)

        # Select the answer with the best score
        if len(answers) > 0:
            best_answer = max(answers, key=lambda x: x["logit_score"])
            predicted_answers.append(
                {"id": example_id, "prediction_text": best_answer["text"]}
            )
        else:
            predicted_answers.append({"id": example_id, "prediction_text": ""})

    theoretical_answers = [{"id": ex["id"], "answers": ex["answers"]} for ex in examples]
    return metric.compute(predictions=predicted_answers, references=theoretical_answers)

In [ ]:

from transformers import TrainingArguments, AutoModelForQuestionAnswering

model = AutoModelForQuestionAnswering.from_pretrained(model_checkpoint)


args = TrainingArguments(
    "bert-finetuned-squad",
    eval_strategy="no",
    save_strategy="epoch",
    learning_rate=2e-5,
    num_train_epochs=1,
    weight_decay=0.01,
    fp16=True,
    push_to_hub=True,
)

from transformers import Trainer

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    processing_class=tokenizer,
)
trainer.train()

In [ ]:
predictions, _, _ = trainer.predict(validation_dataset)
start_logits, end_logits = predictions
compute_metrics(start_logits, end_logits, validation_dataset, raw_datasets["validation"])

In [ ]:
trainer.push_to_hub(commit_message="Training complete")

# traning loop and using fine tume model
<br>
## See [here](https://huggingface.co/learn/llm-course/chapter7/7)

